# Regression - Data Understanding

This notebook queries the Chicago crime dataset, loads a workable analysis sample, and performs preliminary analysis to understand structure, quality, and regression-relevant signals.

In [8]:
from pathlib import Path
from urllib.parse import urlencode
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'regression' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'crimes.csv'
SOCRATA_BASE = 'https://data.cityofchicago.org/resource/ijzp-q8t2.json'
ANALYSIS_SAMPLE_SIZE = 75000

In [9]:
def run_soql(params: dict) -> pd.DataFrame:
    url = f"{SOCRATA_BASE}?{urlencode(params)}"
    return pd.read_json(url)


def load_analysis_sample(limit: int = ANALYSIS_SAMPLE_SIZE) -> pd.DataFrame:
    if DATA_PATH.exists():
        print(f'Loading local file: {DATA_PATH}')
        return pd.read_csv(DATA_PATH, nrows=limit)

    print('Local CSV not found. Loading a sample from the City of Chicago API.')
    params = {
        '$limit': limit,
        '$order': 'date DESC'
    }
    return run_soql(params)


## 1. Dataset-Level Query

Query the live Chicago crime dataset to understand the overall scope before working with a local or sampled dataframe.

In [10]:
dataset_summary = run_soql({
    '$select': 'count(*) as row_count, min(date) as earliest_date, max(date) as latest_date'
})
dataset_summary

,row_count,earliest_date,latest_date
0,8516566,2001-01-01T00:00:00.000,2026-03-14T00:00:00.000


In [4]:
latest_years = run_soql({
    '$select': 'year, count(*) as incidents',
    '$group': 'year',
    '$order': 'year DESC',
    '$limit': 10
})
latest_years

,year,incidents
0,2026,40133
1,2025,236801
2,2024,259034
3,2023,263292
4,2022,240019
5,2021,209665
6,2020,212711
7,2019,261707
8,2018,269155
9,2017,269287


In [5]:
top_primary_types_citywide = run_soql({
    '$select': 'primary_type, count(*) as incidents',
    '$group': 'primary_type',
    '$order': 'incidents DESC',
    '$limit': 15
})
top_primary_types_citywide

,primary_type,incidents
0,THEFT,1809029
1,BATTERY,1551300
2,CRIMINAL DAMAGE,968142
3,NARCOTICS,766493
4,ASSAULT,572377
5,OTHER OFFENSE,531826
6,BURGLARY,449782
7,MOTOR VEHICLE THEFT,438088
8,DECEPTIVE PRACTICE,394470
9,ROBBERY,316500


## 2. Load an Analysis Sample

Use the local CSV when available; otherwise fall back to a recent sample from the API so the notebook remains runnable.

In [6]:
df = load_analysis_sample()
df.head()

Local CSV not found. Loading a sample from the City of Chicago API.


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14136406,JK180685,2026-03-14,023XX N ORCHARD ST,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,1812,18,43,7.00,07,"1,171,210.00","1,915,949.00",2026,2026-03-21T15:47:54.000,41.92,-87.65,"{'latitude': '41.924835846', 'longitude': '-87..."
1,14136488,JK180688,2026-03-14,051XX S WABASH AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,True,225,2,3,40.00,08B,"1,177,590.00","1,870,819.00",2026,2026-03-21T15:47:54.000,41.80,-87.62,"{'latitude': '41.800853616', 'longitude': '-87..."
2,14138063,JK182695,2026-03-14,009XX W SCHOOL ST,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,False,False,1924,19,44,6.00,06,"1,169,236.00","1,922,098.00",2026,2026-03-21T15:47:54.000,41.94,-87.65,"{'latitude': '41.941752159', 'longitude': '-87..."
3,14136281,JK180511,2026-03-14,009XX N CENTRAL PARK AVE,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,1112,11,27,23.00,07,"1,152,212.00","1,905,930.00",2026,2026-03-21T15:47:54.000,41.90,-87.72,"{'latitude': '41.897739256', 'longitude': '-87..."
4,14137838,JK182235,2026-03-14,051XX S MICHIGAN AVE,0810,THEFT,OVER $500,APARTMENT,False,True,225,2,3,40.00,06,"1,178,030.00","1,870,926.00",2026,2026-03-21T15:47:54.000,41.80,-87.62,"{'latitude': '41.801137262', 'longitude': '-87..."


In [7]:
print('Sample shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
df.info()

Sample shape: (75000, 22)

Columns:
['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']

Dtypes:
id                               int64
case_number                        str
date                    datetime64[us]
block                              str
iucr                               str
primary_type                       str
description                        str
location_description               str
arrest                            bool
domestic                          bool
beat                             int64
district                         int64
ward                             int64
community_area                 float64
fbi_code                           str
x_coordinate                   float64
y_coordinate                   float64
year               

## 3. Data Quality Checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})
missing_summary[missing_summary['missing_count'] > 0].head(20)

In [11]:
if 'date' in df.columns and 'Date' not in df.columns:
    df = df.rename(columns={'date': 'Date'})
if 'primary_type' in df.columns and 'Primary Type' not in df.columns:
    df = df.rename(columns={'primary_type': 'Primary Type'})
if 'location_description' in df.columns and 'Location Description' not in df.columns:
    df = df.rename(columns={'location_description': 'Location Description'})
if 'community_area' in df.columns and 'Community Area' not in df.columns:
    df = df.rename(columns={'community_area': 'Community Area'})
if 'domestic' in df.columns and 'Domestic' not in df.columns:
    df = df.rename(columns={'domestic': 'Domestic'})
if 'arrest' in df.columns and 'Arrest' not in df.columns:
    df = df.rename(columns={'arrest': 'Arrest'})
if 'beat' in df.columns and 'Beat' not in df.columns:
    df = df.rename(columns={'beat': 'Beat'})
if 'latitude' in df.columns and 'Latitude' not in df.columns:
    df = df.rename(columns={'latitude': 'Latitude'})
if 'longitude' in df.columns and 'Longitude' not in df.columns:
    df = df.rename(columns={'longitude': 'Longitude'})
if 'year' in df.columns and 'Year' not in df.columns:
    df = df.rename(columns={'year': 'Year'})

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Primary Type'] = df['Primary Type'].astype(str)
df['Location Description'] = df['Location Description'].astype(str)
df['Hour'] = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month

for col in ['Arrest', 'Domestic']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().map({'TRUE': True, 'FALSE': False}).fillna(df[col])

df[['Date', 'Primary Type', 'Location Description', 'Hour', 'DayOfWeek', 'Month']].head()

,Date,Primary Type,Location Description,Hour,DayOfWeek,Month
0,2026-03-14,MOTOR VEHICLE THEFT,STREET,0,Saturday,3
1,2026-03-14,BATTERY,APARTMENT,0,Saturday,3
2,2026-03-14,BURGLARY,STREET,0,Saturday,3
3,2026-03-14,MOTOR VEHICLE THEFT,STREET,0,Saturday,3
4,2026-03-14,THEFT,APARTMENT,0,Saturday,3


## 4. Preliminary Analysis

These quick summaries help identify useful predictors and possible regression targets such as counts by time, area, or location type.

In [12]:
top_primary_types = df['Primary Type'].value_counts().head(15)
top_locations = df['Location Description'].value_counts().head(15)
hourly_counts = df['Hour'].value_counts().sort_index()
daily_counts = df['DayOfWeek'].value_counts().reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

print('Top primary crime types')
display(top_primary_types)
print('\nTop location descriptions')
display(top_locations)
print('\nArrest rate:', df['Arrest'].mean() if df['Arrest'].dtype != object else 'Inspect manually')
print('Domestic rate:', df['Domestic'].mean() if df['Domestic'].dtype != object else 'Inspect manually')

Top primary crime types


Primary Type
THEFT                         16926
BATTERY                       13612
CRIMINAL DAMAGE                8414
ASSAULT                        6587
MOTOR VEHICLE THEFT            6063
OTHER OFFENSE                  5271
DECEPTIVE PRACTICE             4415
BURGLARY                       3535
NARCOTICS                      2294
CRIMINAL TRESPASS              1922
ROBBERY                        1642
WEAPONS VIOLATION              1439
CRIMINAL SEXUAL ASSAULT         533
OFFENSE INVOLVING CHILDREN      502
SEX OFFENSE                     411
Name: count, dtype: int64


Top location descriptions


Location Description
STREET                                    19857
APARTMENT                                 15596
RESIDENCE                                  8665
SIDEWALK                                   2997
SMALL RETAIL STORE                         2816
PARKING LOT / GARAGE (NON RESIDENTIAL)     2764
DEPARTMENT STORE                           1970
RESTAURANT                                 1765
OTHER (SPECIFY)                            1411
ALLEY                                      1344
VEHICLE NON-COMMERCIAL                     1203
COMMERCIAL / BUSINESS OFFICE               1075
RESIDENCE - PORCH / HALLWAY                 960
GAS STATION                                 884
GROCERY FOOD STORE                          845
Name: count, dtype: int64


Arrest rate: 0.15526666666666666
Domestic rate: 0.19516


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

top_primary_types.sort_values().plot(kind='barh', ax=axes[0, 0], title='Top Crime Types')
axes[0, 0].set_xlabel('Count')

top_locations.sort_values().plot(kind='barh', ax=axes[0, 1], title='Top Crime Locations')
axes[0, 1].set_xlabel('Count')

hourly_counts.plot(kind='bar', ax=axes[1, 0], title='Crime by Hour')
axes[1, 0].set_xlabel('Hour')
axes[1, 0].set_ylabel('Count')

daily_counts.plot(kind='bar', ax=axes[1, 1], title='Crime by Day of Week')
axes[1, 1].set_xlabel('Day')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
candidate_regression_views = {
    'monthly_incident_volume': df.groupby(df['Date'].dt.to_period('M')).size().head(),
    'beat_hour_counts': df.groupby(['Beat', 'Hour']).size().sort_values(ascending=False).head(),
    'community_area_counts': df.groupby('Community Area').size().sort_values(ascending=False).head()
}

candidate_regression_views

## 5. Initial Takeaways

- Missingness should be reviewed closely for geographic and community-level fields.
- `Date` can be expanded into hour, weekday, month, and time-window features.
- A regression task can be framed as predicting incident volume by beat, community area, or time bucket.
- `Primary Type`, `Location Description`, `Beat`, `Arrest`, and `Domestic` look like strong candidate features for downstream analysis.